## 📊 Data Preprocessing & Feature Engineering Project — Healthcare Dataset


# Part A: Handling Missing Values

## 🔹 Step 1: Load Dataset

In [1]:
import numpy as np
import pandas as pd

data = pd.read_csv("healthcare_dataset_700_rows.csv")
data.head()

,patient_id,age,gender,region,bmi,blood_pressure,cholesterol,glucose,disease_risk
0,100001,32.0,Male,East,26.1,131.9,251.0,132.6,0
1,100002,87.0,Male,West,27.7,123.6,210.2,142.8,1
2,100003,89.0,Male,West,27.0,145.6,206.5,102.0,1
3,100004,38.0,Female,East,25.5,134.3,206.4,92.0,0
4,100005,66.0,Male,East,29.1,102.0,224.9,62.6,0


## 🔹 Step 2: Identify Missing Values
Summary report of missing values per column (count and percentage).

In [2]:
data.isnull().sum()

patient_id         0
age               35
gender            28
region            28
bmi               20
blood_pressure     0
cholesterol       30
glucose           30
disease_risk       0
dtype: int64

In [3]:
data.isnull().mean() * 100

patient_id        0.000000
age               5.000000
gender            4.000000
region            4.000000
bmi               2.857143
blood_pressure    0.000000
cholesterol       4.285714
glucose           4.285714
disease_risk      0.000000
dtype: float64

## 🔹 Step 3: Simple Imputer (Numerical)
Replace missing `bmi` and `cholesterol` with the column **mean**.

In [4]:
data_simple = data.copy()

data_simple["bmi"] = data_simple["bmi"].fillna(data_simple["bmi"].mean())
data_simple["cholesterol"] = data_simple["cholesterol"].fillna(data_simple["cholesterol"].mean())

print("Missing values after Simple Imputation:")
print(data_simple[["bmi", "cholesterol"]].isnull().sum())

data_simple.head()

Missing values after Simple Imputation:
bmi            0
cholesterol    0
dtype: int64


,patient_id,age,gender,region,bmi,blood_pressure,cholesterol,glucose,disease_risk
0,100001,32.0,Male,East,26.1,131.9,251.0,132.6,0
1,100002,87.0,Male,West,27.7,123.6,210.2,142.8,1
2,100003,89.0,Male,West,27.0,145.6,206.5,102.0,1
3,100004,38.0,Female,East,25.5,134.3,206.4,92.0,0
4,100005,66.0,Male,East,29.1,102.0,224.9,62.6,0


## In my dataset catagorical column dosnt hve any missing value so i cant perform simpal imputer on catogarical data .
but we can handel or replace missing values using most frequnt valus if data have missing values 

## 🔹 Step 4: Most Frequent Imputation (Categorical)
Replace missing `gender` and `region` with the most frequent category (mode).

In [5]:
data_mode = data.copy()

data_mode["gender"] = data_mode["gender"].fillna(data_mode["gender"].mode()[0])
data_mode["region"] = data_mode["region"].fillna(data_mode["region"].mode()[0])

print("Missing values after Mode Imputation:")
print(data_mode[["gender", "region"]].isnull().sum())

data_mode.head()

Missing values after Mode Imputation:
gender    0
region    0
dtype: int64


,patient_id,age,gender,region,bmi,blood_pressure,cholesterol,glucose,disease_risk
0,100001,32.0,Male,East,26.1,131.9,251.0,132.6,0
1,100002,87.0,Male,West,27.7,123.6,210.2,142.8,1
2,100003,89.0,Male,West,27.0,145.6,206.5,102.0,1
3,100004,38.0,Female,East,25.5,134.3,206.4,92.0,0
4,100005,66.0,Male,East,29.1,102.0,224.9,62.6,0


## 🔹 Step 5: Missing Indicator + Random Sample Imputation
Create a binary indicator column for missingness in `bmi`, then fill missing values by randomly sampling from existing observed values.

In [6]:
data_random = data.copy()

data_random["bmi_missing"] = data_random["bmi"].isnull().astype(int)

data_random["bmi"] = data_random["bmi"].apply(
    lambda x: data_random["bmi"].dropna().sample(1, random_state=42).values[0] if pd.isnull(x) else x
)

print("Missing values after Random Sampling:")
print(data_random["bmi"].isnull().sum())

data_random.head()

Missing values after Random Sampling:
0


,patient_id,age,gender,region,bmi,blood_pressure,cholesterol,glucose,disease_risk,bmi_missing
0,100001,32.0,Male,East,26.1,131.9,251.0,132.6,0,0
1,100002,87.0,Male,West,27.7,123.6,210.2,142.8,1,0
2,100003,89.0,Male,West,27.0,145.6,206.5,102.0,1,0
3,100004,38.0,Female,East,25.5,134.3,206.4,92.0,0,0
4,100005,66.0,Male,East,29.1,102.0,224.9,62.6,0,0


## 🔹 Step 6: KNN Imputer
Multivariate imputation using k-Nearest Neighbors on the numerical columns (`age`, `bmi`, `cholesterol`, `glucose`).

In [7]:
from sklearn.impute import KNNImputer

data_knn = data.copy()
num_cols = ["age", "bmi", "cholesterol", "glucose"]

knn = KNNImputer(n_neighbors=5)
data_knn[num_cols] = knn.fit_transform(data_knn[num_cols])

print("Missing values after KNN:")
print(data_knn[num_cols].isnull().sum())

data_knn.head()

Missing values after KNN:
age            0
bmi            0
cholesterol    0
glucose        0
dtype: int64


,patient_id,age,gender,region,bmi,blood_pressure,cholesterol,glucose,disease_risk
0,100001,32.0,Male,East,26.1,131.9,251.0,132.6,0
1,100002,87.0,Male,West,27.7,123.6,210.2,142.8,1
2,100003,89.0,Male,West,27.0,145.6,206.5,102.0,1
3,100004,38.0,Female,East,25.5,134.3,206.4,92.0,0
4,100005,66.0,Male,East,29.1,102.0,224.9,62.6,0


## 🔹 Step 7: MICE Algorithm (Multiple Imputation by Chained Equations)
Performs chained equations imputation across the numerical columns together, modeling each column as a function of the others.

In [8]:
from sklearn.experimental import enable_iterative_imputer
from sklearn.impute import IterativeImputer

data_mice = data.copy()

mice = IterativeImputer(random_state=42)
data_mice[num_cols] = mice.fit_transform(data_mice[num_cols])

print("Missing values after MICE:")
print(data_mice[num_cols].isnull().sum())

data_mice.head()

Missing values after MICE:
age            0
bmi            0
cholesterol    0
glucose        0
dtype: int64


,patient_id,age,gender,region,bmi,blood_pressure,cholesterol,glucose,disease_risk
0,100001,32.0,Male,East,26.1,131.9,251.0,132.6,0
1,100002,87.0,Male,West,27.7,123.6,210.2,142.8,1
2,100003,89.0,Male,West,27.0,145.6,206.5,102.0,1
3,100004,38.0,Female,East,25.5,134.3,206.4,92.0,0
4,100005,66.0,Male,East,29.1,102.0,224.9,62.6,0


Categorical columns (`gender`, `region`) are filled using mode imputation so the dataset has **zero missing values overall**.

In [9]:
data_mice["gender"] = data_mice["gender"].fillna(data_mice["gender"].mode()[0])
data_mice["region"] = data_mice["region"].fillna(data_mice["region"].mode()[0])

print("Total missing values remaining:")
data_mice.isnull().sum()

Total missing values remaining:


patient_id        0
age               0
gender            0
region            0
bmi               0
blood_pressure    0
cholesterol       0
glucose           0
disease_risk      0
dtype: int64

# Part B: Handling Outliers

## 🔹 Step 8: Z-Score Method
Identify and remove rows where any numerical column has a z-score with absolute value ≥ 3.

In [10]:
from scipy.stats import zscore

clean_data = data_mice.copy()

z = np.abs(zscore(clean_data[num_cols]))
data_z = clean_data[(z < 3).all(axis=1)]

print("Original Shape:", clean_data.shape)
print("After Z-score:", data_z.shape)

data_z.head()

Original Shape: (700, 9)
After Z-score: (663, 9)


,patient_id,age,gender,region,bmi,blood_pressure,cholesterol,glucose,disease_risk
0,100001,32.0,Male,East,26.1,131.9,251.0,132.6,0
1,100002,87.0,Male,West,27.7,123.6,210.2,142.8,1
2,100003,89.0,Male,West,27.0,145.6,206.5,102.0,1
3,100004,38.0,Female,East,25.5,134.3,206.4,92.0,0
4,100005,66.0,Male,East,29.1,102.0,224.9,62.6,0


## 🔹 Step 9: IQR Method
Use the interquartile range to flag and remove unusual values across the numerical columns.

In [11]:
Q1 = clean_data[num_cols].quantile(0.25)
Q3 = clean_data[num_cols].quantile(0.75)
IQR = Q3 - Q1

data_iqr = clean_data[~((clean_data[num_cols] < (Q1 - 1.5*IQR)) |
                        (clean_data[num_cols] > (Q3 + 1.5*IQR))).any(axis=1)]

print("After IQR:", data_iqr.shape)

data_iqr.head()

After IQR: (635, 9)


,patient_id,age,gender,region,bmi,blood_pressure,cholesterol,glucose,disease_risk
0,100001,32.0,Male,East,26.1,131.9,251.0,132.6,0
1,100002,87.0,Male,West,27.7,123.6,210.2,142.8,1
2,100003,89.0,Male,West,27.0,145.6,206.5,102.0,1
3,100004,38.0,Female,East,25.5,134.3,206.4,92.0,0
4,100005,66.0,Male,East,29.1,102.0,224.9,62.6,0


## 🔹 Step 10: Percentile Capping
Cap values below the 1st percentile and above the 99th percentile instead of removing rows.

In [12]:
lower = clean_data[num_cols].quantile(0.01)
upper = clean_data[num_cols].quantile(0.99)

data_pct = clean_data.copy()
data_pct[num_cols] = data_pct[num_cols].clip(lower, upper, axis=1)

print("After Percentile Capping:")
data_pct.describe()

After Percentile Capping:


,patient_id,age,bmi,blood_pressure,cholesterol,glucose,disease_risk
count,700.00000,700.000000,700.000000,700.00000,700.000000,700.000000,700.000000
mean,100350.50000,54.100887,26.439215,123.53600,210.516307,110.164600,0.265714
std,202.21688,21.086315,5.725243,25.85143,51.414795,44.517203,0.442029
min,100001.00000,18.000000,16.992000,72.40000,127.991000,37.393000,0.000000
25%,100175.75000,35.000000,23.300000,108.37500,182.800000,87.450000,0.000000
50%,100350.50000,54.087138,26.100000,122.35000,207.900000,107.900000,0.000000
75%,100525.25000,72.000000,28.200000,134.50000,227.225000,125.550000,1.000000
max,100700.00000,90.000000,57.401000,296.40000,527.272000,388.097000,1.000000


## 🔹 Step 11: Winsorization
Cap the most extreme 5% of values on each tail instead of removing them.

In [16]:
from scipy.stats.mstats import winsorize

data_win = clean_data.copy()

for col in num_cols:
    data_win[col] = winsorize(data_win[col], limits=[0.05, 0.05])

print("After Winsorization:")
data_win.describe()

import warnings
warnings.filterwarnings("ignore", category=UserWarning)

After Winsorization:


# Part C: Final Clean Dataset

## 🔹 Step 12: Build the Final Cleaned Dataset
Combining the best techniques identified in the comparison below — **MICE** for missing-value imputation and **IQR** for outlier removal — to produce the final, machine-learning-ready dataset.

In [14]:
final_data = data_iqr.copy()
final_data.to_csv("cleaned_health_data.csv", index=False)

print("Final dataset created")

final_data.head()

Final dataset created


,patient_id,age,gender,region,bmi,blood_pressure,cholesterol,glucose,disease_risk
0,100001,32.0,Male,East,26.1,131.9,251.0,132.6,0
1,100002,87.0,Male,West,27.7,123.6,210.2,142.8,1
2,100003,89.0,Male,West,27.0,145.6,206.5,102.0,1
3,100004,38.0,Female,East,25.5,134.3,206.4,92.0,0
4,100005,66.0,Male,East,29.1,102.0,224.9,62.6,0


## 🔹 Before vs After Comparison

In [15]:
print("Original Shape:", data.shape)
print("Final Shape:", final_data.shape)

print("\n🔹 Original Data Summary")
print(data.describe())

print("\n🔹 Cleaned Data Summary")
print(final_data.describe())

Original Shape: (700, 9)
Final Shape: (635, 9)

🔹 Original Data Summary
         patient_id         age         bmi  blood_pressure  cholesterol  \
count     700.00000  665.000000  680.000000       700.00000    670.00000   
mean   100350.50000   54.100752   26.476471       123.53600    210.81209   
std       202.21688   21.634917    6.233742        25.85143     55.71210   
min    100001.00000   18.000000   12.100000        72.40000    100.50000   
25%    100175.75000   35.000000   23.200000       108.37500    181.47500   
50%    100350.50000   53.000000   26.000000       122.35000    205.75000   
75%    100525.25000   73.000000   28.300000       134.50000    228.97500   
max    100700.00000   90.000000   67.400000       296.40000    607.80000   

          glucose  disease_risk  
count  670.000000    700.000000  
mean   110.866119      0.265714  
std     51.002778      0.442029  
min     23.900000      0.000000  
25%     86.625000      0.000000  
50%    105.650000      0.000000  
75%  

## 📌 Final Observations

### ✔ Best Imputation Method
**MICE** performed best as it models each numerical column as a function of the others, preserving relationships between variables better than mean or mode imputation alone.

### ✔ Best Outlier Method
**IQR** effectively identified and removed extreme values without assuming a normal distribution, making it robust for slightly skewed medical measurements like cholesterol and glucose.
